In [35]:
# Identify athletes meeting new Dec 24 OCTC selection rules: 

# OCTC qualification period is 1 Jan 2024 to current date

#1. Segregate into Male and Femal 
#2. For each gender perform the following: 
#a. Sort data by mapped eent, then perf scalar (higher the better)
#b. Identify tiers based on performance - Tier 1 is meets bronze medal mark for SEAG, Tier 2 is 2% and Tier 3 is 3.5%
#c. Check - if athlete met bronze or 2%/3.5% then delta_benchmark is zero or +, delta2% is + and delta 3.5% is +
#d. Top ranked athletes for each event are chosen. Max number of athletes for each event is 3, except for 100m/400m which is 6
#    This includes athletes on spex scholarship and potential
#e. The max for each tier is 2. Lower ranked athletes move down one tier.
#3. If athlete qualifies for more than one event the higher tier event is given
#4. Jump and throws junior program to be solved separately

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [36]:
# Import usual modules
import pandas as pd
import csv
import math
import os
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import openpyxl
import datetime
from scipy.stats import lognorm
import re
import string
from bs4 import BeautifulSoup
import requests
import unicodedata # for removing accented characters
import datetime
import icecream as ic
import dateutil.parser as parser 
import datacompy
import pytz



from google.cloud import storage



In [37]:
# PRODUCTION ENVIRONMENT
# Extract timed event records

import pandas_gbq
from google.oauth2 import service_account

credentials = service_account.Credentials.from_service_account_file(
    '/Users/veesheenyuen/Desktop/DataScience/Keys/saa-analytics-7c8937b70609.json',
    
    
)

sql1="""
SELECT NAME, RESULT, TEAM, AGE, RANK AS COMPETITION_RANK, STAGE, DIVISION, EVENT, DISTANCE, EVENT_CLASS, UNIQUE_ID, DOB, NATIONALITY, WIND, CATEGORY_EVENT, GENDER, COMPETITION, DATE, YEAR, REGION, TIMESTAMP
FROM `saa-analytics.results.PRODUCTION`
"""

competitors = pandas_gbq.read_gbq(sql1, project_id="saa-analytics", credentials=credentials)




Downloading: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████|


In [38]:
competitors

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,STAGE,DIVISION,EVENT,DISTANCE,EVENT_CLASS,...,DOB,NATIONALITY,WIND,CATEGORY_EVENT,GENDER,COMPETITION,DATE,YEAR,REGION,TIMESTAMP
0,Zhe Xi Ho,11.50,,,1.0,,,100m,,,...,6 May 03,SGP,0.8,Sprint,Male,BFTTA Open,2026-05-17 00:00:00+00:00,2026,International,2026-05-27 18:31:00+00:00
1,Xiuying Hu,3:32:42,,,67.0,,,Marathon,,,...,8 Oct 78,SGP,,Marathon,Female,Helsinki City Run,2026-05-16 00:00:00+00:00,2026,International,2026-05-27 18:31:00+00:00
2,Kampton Kam,2.15,,,2.0,,,High Jump,,,...,6 Mar 01,SGP,,Jump,Male,Ivy League Heptagonal Outdoor Track & Field Ch...,2026-05-17 00:00:00+00:00,2026,International,2026-05-27 18:31:00+00:00
3,Xander Ho Ann Heng,10.61,,,2.0,Heats,,100m,,,...,19 May 00,SGP,0.9,Sprint,Male,Regional Athletics Invitational Meeting,2026-05-14 00:00:00+00:00,2026,International,2026-05-27 18:31:00+00:00
4,Xander Ho Ann Heng,21.56,,,1.0,Heats,,200m,,,...,19 May 00,SGP,0.4,Sprint,Male,Regional Athletics Invitational Meeting,2026-05-15 00:00:00+00:00,2026,International,2026-05-27 18:31:00+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
189704,Jun Rui Julius Koo,6.67,Club Zoom - Energetix Pte Ltd 2,,14,Final,,Turbojav Throw,,nan,...,2014-08-05,SGP,nan,Throw,Male,Pesta Sukan Athletics 2025,2025-07-26 14:30:00+00:00,2025,Local,2025-12-10 13:45:00+00:00
189705,Kevin Rafael Theng,11.37,- Energetix Pte Ltd 2,,11,Final,,Turbojav Throw,,nan,...,2014-04-08,SGP,nan,Throw,Male,Pesta Sukan Athletics 2025,2025-07-26 14:30:00+00:00,2025,Local,2025-12-10 13:45:00+00:00
189706,Lucas Wang,22.97,ActiveSG Academies & Club - Energetix Pte Ltd 2,,3,Final,,Turbojav Throw,,nan,...,NaT,USA,nan,Throw,Male,Pesta Sukan Athletics 2025,2025-07-26 14:30:00+00:00,2025,Local,2025-12-10 13:45:00+00:00
189707,Theodore Mosbergen Soo,15.25,- Energetix Pte Ltd 2,,8,Final,,Turbojav Throw,,nan,...,2014-10-29,SGP,nan,Throw,Male,Pesta Sukan Athletics 2025,2025-07-26 14:30:00+00:00,2025,Local,2025-12-10 13:45:00+00:00


In [1]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/Marathon/')

competitors.to_csv('all.csv', sep=',', encoding='utf-8-sig', index=False)

NameError: name 'os' is not defined

In [39]:
#os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/OCTC/')

#unique_list = pd.Series(list(set(competitors['EVENT_CLASS'])))

#unique_list.to_csv('unique_event_class.csv', index=False)

In [40]:
competitors

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,STAGE,DIVISION,EVENT,DISTANCE,EVENT_CLASS,...,DOB,NATIONALITY,WIND,CATEGORY_EVENT,GENDER,COMPETITION,DATE,YEAR,REGION,TIMESTAMP
0,Zhe Xi Ho,11.50,,,1.0,,,100m,,,...,6 May 03,SGP,0.8,Sprint,Male,BFTTA Open,2026-05-17 00:00:00+00:00,2026,International,2026-05-27 18:31:00+00:00
1,Xiuying Hu,3:32:42,,,67.0,,,Marathon,,,...,8 Oct 78,SGP,,Marathon,Female,Helsinki City Run,2026-05-16 00:00:00+00:00,2026,International,2026-05-27 18:31:00+00:00
2,Kampton Kam,2.15,,,2.0,,,High Jump,,,...,6 Mar 01,SGP,,Jump,Male,Ivy League Heptagonal Outdoor Track & Field Ch...,2026-05-17 00:00:00+00:00,2026,International,2026-05-27 18:31:00+00:00
3,Xander Ho Ann Heng,10.61,,,2.0,Heats,,100m,,,...,19 May 00,SGP,0.9,Sprint,Male,Regional Athletics Invitational Meeting,2026-05-14 00:00:00+00:00,2026,International,2026-05-27 18:31:00+00:00
4,Xander Ho Ann Heng,21.56,,,1.0,Heats,,200m,,,...,19 May 00,SGP,0.4,Sprint,Male,Regional Athletics Invitational Meeting,2026-05-15 00:00:00+00:00,2026,International,2026-05-27 18:31:00+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
189704,Jun Rui Julius Koo,6.67,Club Zoom - Energetix Pte Ltd 2,,14,Final,,Turbojav Throw,,nan,...,2014-08-05,SGP,nan,Throw,Male,Pesta Sukan Athletics 2025,2025-07-26 14:30:00+00:00,2025,Local,2025-12-10 13:45:00+00:00
189705,Kevin Rafael Theng,11.37,- Energetix Pte Ltd 2,,11,Final,,Turbojav Throw,,nan,...,2014-04-08,SGP,nan,Throw,Male,Pesta Sukan Athletics 2025,2025-07-26 14:30:00+00:00,2025,Local,2025-12-10 13:45:00+00:00
189706,Lucas Wang,22.97,ActiveSG Academies & Club - Energetix Pte Ltd 2,,3,Final,,Turbojav Throw,,nan,...,NaT,USA,nan,Throw,Male,Pesta Sukan Athletics 2025,2025-07-26 14:30:00+00:00,2025,Local,2025-12-10 13:45:00+00:00
189707,Theodore Mosbergen Soo,15.25,- Energetix Pte Ltd 2,,8,Final,,Turbojav Throw,,nan,...,2014-10-29,SGP,nan,Throw,Male,Pesta Sukan Athletics 2025,2025-07-26 14:30:00+00:00,2025,Local,2025-12-10 13:45:00+00:00


In [78]:
marathon_df = competitors[competitors["EVENT"].isin(["Marathon", "Half Marathon"])].copy()

In [79]:
marathoners=marathon_df[((marathon_df['YEAR']=='2025')|(marathon_df['YEAR']=='2026'))]

In [80]:
# Convert timing string into sortable duration
marathoners["RESULT_TD"] = pd.to_timedelta(marathoners["RESULT"], errors="coerce")

# Use seconds for ranking/sorting
marathoners["RESULT_SECONDS"] = marathoners["RESULT_TD"].dt.total_seconds()


/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_84695/3756448205.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  marathoners["RESULT_TD"] = pd.to_timedelta(marathoners["RESULT"], errors="coerce")
/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_84695/3756448205.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  marathoners["RESULT_SECONDS"] = marathoners["RESULT_TD"].dt.total_seconds()


In [81]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/Marathon/')

marathoners.to_excel('marathoners.xlsx')

In [57]:
# Remove rows where timing could not be parsed

marathoners = marathoners.dropna(subset=["RESULT_TIME"])

In [82]:
# Rank separately by event
marathoners["EVENT_RANK"] = (
    marathoners
    .groupby("EVENT")["RESULT_SECONDS"]
    .rank(method="min", ascending=True)
    .astype("Int64")
)

# Sort fastest to slowest within each event
marathon_ranked = marathoners.sort_values(
    ["EVENT", "EVENT_RANK", "RESULT_SECONDS", "NAME"]
)



/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_84695/2041517799.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  marathoners["EVENT_RANK"] = (


In [83]:
display_cols = [
    "EVENT_RANK",
    "NAME",
    "RESULT",
    "GENDER",
    "TEAM",
    "COMPETITION",
    "DATE",
    "NATIONALITY"
]

half_marathon_ranked = marathon_ranked[
    marathon_ranked["EVENT"] == "Half Marathon"
][display_cols]

marathon_ranked_only = marathon_ranked[
    marathon_ranked["EVENT"] == "Marathon"
][display_cols]

In [84]:
marathon_ranked_only

,EVENT_RANK,NAME,RESULT,GENDER,TEAM,COMPETITION,DATE,NATIONALITY
165420,1,Rui Yong Soh,02:27:51,Male,,Chevron Houston Marathon,2025-01-19 00:00:00+00:00,SGP
165531,2,Caleb Hia,2:29:21,Male,<NA>,41th Estra Firenze Marathon,2025-11-30 00:00:00+00:00,SGP
165528,3,Yong He,02:29:27,Male,<NA>,Beijing Marathon,2025-11-02 00:00:00+00:00,SGP
165529,4,"HE, YONG HENRY",02:29:43,Male,,Beijing Marathon,2025-11-02 00:00:00+00:00,
165530,4,"HE, YONG HENRY",02:29:43,Male,Individual,Beijing Marathon,2025-11-02 00:00:00+00:00,SGP
...,...,...,...,...,...,...,...,...
165492,208,"Foo, Ray",07:46:37,Male,,London Marathon 2025,2025-04-27 00:00:00+00:00,SGP
165470,<NA>,"Kunasakaran, Karthig",-,Male,,London Marathon 2025,2025-04-27 00:00:00+00:00,SGP
165456,<NA>,"Lee, Jason",-,Male,,London Marathon 2025,2025-04-27 00:00:00+00:00,SGP
165441,<NA>,"Tan, Cheryl",-,Female,,London Marathon 2025,2025-04-27 00:00:00+00:00,SGP


In [85]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/Marathon/')

with pd.ExcelWriter("marathon_ranked.xlsx", engine="xlsxwriter") as writer:
    half_marathon_ranked.to_excel(writer, sheet_name="Half Marathon", index=False)
    marathon_ranked_only.to_excel(writer, sheet_name="Marathon", index=False)